# Business Analytics Project

This notebook demonstrates exploratory data analysis (EDA) and predictive modelling on a synthetic business dataset. The dataset simulates customer transactions across multiple product categories and geographic regions.

In [ ]:
import pandas as pd

# Load the synthetic dataset
data = pd.read_csv('synthetic_business_data.csv')

# Inspect the first few rows
data.head()

## Summary statistics

We'll start by examining summary statistics for the numerical columns of the dataset.

In [ ]:
# Summary statistics for numeric fields
data.describe(include='all')

## Exploratory Data Analysis

Next we explore the distribution of customer ages as well as relationships between numerical features.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

# Age distribution
plt.figure(figsize=(8, 4))
sns.histplot(data['Age'], bins=20, kde=True, color='steelblue')
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

In [ ]:
# Correlation matrix for numeric variables
numeric_cols = ['Age', 'UnitsPurchased', 'UnitPrice', 'MarketingSpend', 'Revenue']
corr = data[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='viridis')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Total revenue by product category
plt.figure(figsize=(10, 5))
category_revenue = data.groupby('ProductCategory')['Revenue'].sum().sort_values(ascending=False)
sns.barplot(x=category_revenue.index, y=category_revenue.values, palette='muted')
plt.title('Total Revenue by Product Category')
plt.xlabel('Product Category')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.show()

## Predictive Modelling

We build two predictive models: a regression model to predict revenue and a classification model to identify high purchase events. Categorical variables are one‑hot encoded and the data is split into training and test sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LinearRegression, LogisticRegression

# Separate features and targets
X = data.drop(columns=['Revenue', 'HighPurchase', 'Date', 'CustomerID'])
y_reg = data['Revenue']
y_clf = data['HighPurchase']

# Identify categorical and numeric columns
cat_cols = ['Gender', 'Region', 'ProductCategory']
num_cols = ['Age', 'UnitsPurchased', 'UnitPrice', 'MarketingSpend']

# Preprocessing: one-hot encode categoricals, keep numerics as is
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('numeric', 'passthrough', num_cols)
])

# Split data for regression
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
# Split data for classification
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42)

In [ ]:
# Regression pipeline
reg_model = Pipeline(steps=[('preprocess', preprocessor), ('model', LinearRegression())])
reg_model.fit(X_train_reg, y_train_reg)

# Predict on test data
y_pred_reg = reg_model.predict(X_test_reg)

# Evaluate regression model
mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
print(f'Mean Squared Error: {mse:.2f}')
print(f'R^2 Score: {r2:.4f}')

# Plot actual vs. predicted revenue
plt.figure(figsize=(8, 5))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.6)
plt.xlabel('Actual Revenue')
plt.ylabel('Predicted Revenue')
plt.title('Actual vs. Predicted Revenue (Regression)')
plt.show()

In [ ]:
# Classification pipeline
clf_model = Pipeline(steps=[('preprocess', preprocessor), ('model', LogisticRegression(max_iter=1000))])
clf_model.fit(X_train_clf, y_train_clf)

# Predict on test data
y_pred_clf = clf_model.predict(X_test_clf)

# Evaluate classification model
acc = accuracy_score(y_test_clf, y_pred_clf)
cm = confusion_matrix(y_test_clf, y_pred_clf)
print(f'Accuracy: {acc:.4f}')
print('Confusion Matrix:')
print(cm)

print('Classification Report:')
print(classification_report(y_test_clf, y_pred_clf))

## Conclusion

In this notebook we explored a synthetic business dataset, created visual summaries and built simple predictive models. The regression model achieved a high R² because revenue is directly derived from units purchased and unit price, while the classification model identifies high purchase events with reasonable accuracy. Feel free to explore alternative models, feature engineering strategies or other analysis questions.